<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [6]</a>'.</span>

<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 4: *Variable Selection*
##### Version Number: 4.0
---
### Contents  
> 1. *Water Demand*
> 2. *Water Supply*
> 3. *Water Supply Indexes*
> 4. *Fire Danger Indicators*
> 5. *Social Variables*
> 6. *Temporal and Geographic Varaibles*
> 7. *Export File*
---
### Notes
- This module visualizes key variables to assess their relationships with wildfire severity categories. Based on the `Categorical` target, we explore how different weather features interact and correlate with fire risk.
---
### Inputs
- `engineered_samples.csv` engineered and cleaned samples data with weather, fire, and grid data.
---
### Outputs 
- `X`,`y`,`details` - Split training data filtered from 2018 to 2024
- `pal_x`, `pal_y`, `pal_details` - Split training data from 2025 for case study
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime
import json

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

# Set consistent plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

---

In [3]:
def plot_all(df, target1, target2, target3, title):
    grid_kde(df)

### Loading Data

In [4]:
samples = pd.read_csv('../data/processed/engineered_samples.csv')

---

## Split dataset temporarily for variable analysis

In [5]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date', 'grid_id',
       'geometry','area_in_cali','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x','centroid_northing','centroid_easting','Target_Damage','Target_Ignition',
               'Target_Spread','Year','State Region']

coded_columns = ['dominant_province_description','dominant_section_description','Season']

numerical_data = samples.drop(columns=text_columns + coded_columns)
detail_data = samples[text_columns]

target_ignition = samples['Target_Ignition']
target_spread = samples['Target_Spread']
target_damage = samples['Target_Damage']

## Scale numerical columns for easier side by side comparisons

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [6]:
scaler = MinMaxScaler()

# Scale main dataset
X_scaled = scaler.fit_transform(numerical_data)
X = pd.DataFrame(X_scaled, columns=numerical_data.columns, index=numerical_data.index)

X = pd.concat([X,samples[coded_columns]],axis=1)

ValueError: could not convert string to float: 'Southern'

## DIrect Water Demand Indicators

In [ ]:
water_demand = [
    "Actual Evapotranspiration",
    "Solar Radiation",
    "Daily Minimum Air Temperature",
    "Daily Maximum Air Temperature",
    "Vapor Pressure Deficit",
    "Wind Speed",
]

In [ ]:
plot_all(X[water_demand], target_ignition, target_spread,target_damage, 'Water Demand')

---

## Water Supply Indicators

In [ ]:
water_supply = [
    "log_Precipitation",
    "Maximum Relative Humidity",
    "Minimum Relative Humidity",
    "Specific Humidity",
    "100-hour Dead Fuel Moisture",
    "1000-hour Dead Fuel Moisture"
]

In [ ]:
plot_all(X[water_supply], target_ignition, target_spread,target_damage,'Water Supply')

---

## Water Supply Indexes

In [ ]:
water_supply_indexes = ["SPI 30-Day",
    "SPI 180-Day",
    "SPEI 30-Day",
    "SPEI 90-Day",
    "SPEI 180-Day",
    "Palmer Drought Severity Index"
]

In [ ]:
plot_all(X[water_supply_indexes], target_ignition, target_spread,target_damage,'Water Supply Indexes')

## Fire Danger

In [ ]:
fire_danger = [
    "Burning Index",
    "Energy Release Component",
    'Santa_Ana_Score',
    'fire_count']

In [ ]:
plot_all(X[fire_danger], target_ignition, target_spread,target_damage,'Fire Danger')

## Social Variables

In [ ]:
social = ['log_total_housing', 'log_total_population',
       'log_housing_density', 'log_population_density', 'median_income']

In [ ]:
plot_all(X[social], target_ignition, target_spread,target_damage,'Social')

## Infrastructure

In [ ]:
infrastructure = ['power_line_meters','power_line_density','road_length_meters','road_density']

In [ ]:
plot_all(X[infrastructure], target_ignition, target_spread,target_damage,'Infrastructure')

## Elevation

In [ ]:
elevation = ['elevation_range', 'elevation_mean',
       'elevation_std', 'slope_max', 'slope_range', 'slope_mean', 'slope_std',
       'northness_mean', 'eastness_mean']

In [ ]:
plot_all(X[elevation], target_ignition, target_spread,target_damage,'Elevation')

## WUI

In [ ]:
WUI = ['influence_zone', 'interface_zone', 'intermix_zone']

In [ ]:
plot_all(X[WUI], target_ignition, target_spread,target_damage,'WUI')

## Ecological

In [ ]:
ecoregion = ['dominant_province_percent', 'sum_province_area', 'sum_section_area',
       'dominant_section_percent']

In [ ]:
plot_all(X[ecoregion], target_ignition, target_spread,target_damage,'Ecological')

## Land Cover

In [ ]:
land_cover = ['forest_percent','developed_percent', 'other_percent', 'shrub_grass_percent',
       'wetlands_percent']

In [ ]:
plot_all(X[land_cover], target_ignition, target_spread,target_damage,'Land Cover')

---

In [ ]:
interactions = ['Wind Speed_x_100-hour Dead Fuel Moisture',
 'Vapor Pressure Deficit_x_Solar Radiation',
 'log_Precipitation_x_1000-hour Dead Fuel Moisture',
 'northness_mean_x_Daily Maximum Air Temperature',
 'road_density_x_forest_percent',
 'power_line_density_x_log_total_housing']

In [ ]:
plot_all(X[interactions], target_ignition, target_spread,target_damage,'Interactions')

## Wind Slope Interaction

In [ ]:
wind_slope = ['Wind Speed_x_slope_mean',
 'Wind Speed_x_slope_max',
 'Wind Speed_x_northness_mean',
 'Wind Speed_x_eastness_mean',
 'Wind Speed_x_elevation_mean',
 'Wind Speed_x_elevation_range',
 'Wind Speed 7 Day Median_x_slope_mean',
 'Wind Speed 7 Day Median_x_slope_max',
 'Wind Speed 7 Day Median_x_northness_mean',
 'Wind Speed 7 Day Median_x_eastness_mean',
 'Wind Speed 7 Day Median_x_elevation_mean',
 'Wind Speed 7 Day Median_x_elevation_range']

In [ ]:
plot_all(X[wind_slope], target_ignition, target_spread,target_damage,'Wind Slope Interactions')

## Spatial Features

In [ ]:
spatial = ['avg_dist_to_all_reservoirs_same_day','dist_to_reservoir_same_day',
           'dist_to_fires_same_day','avg_dist_to_fires_same_day',
           'days_since_last_fire']

samples[spatial] = np.sqrt(samples[spatial])

In [ ]:
plot_all(X[spatial], target_ignition, target_spread,target_damage,'Spatial')

## Other Features

In [ ]:
others = [
    "NDVI_mean_difference",
    "NDVI_mean_difference_has_value",
    'reservoir_count',
    'log_total_reservoir_level',
    'stations_missing_levels'
]

In [ ]:
plot_all(X[others], target_ignition, target_spread,target_damage,'Wind Slope Interactions')

In [ ]:
coded_ecoregion = [
    # Province
    "dominant_province_description_American Semi-Desert and Desert",
    "dominant_province_description_California Coastal Chaparral Forest and Shrub",
    "dominant_province_description_California Coastal Range Open Woodland-Shrub-Coniferous Forest-Meadow",
    "dominant_province_description_California Coastal Steppe-Mixed Forest-Redwood Forest",
    "dominant_province_description_California Dry Steppe",
    "dominant_province_description_Intermountain Semi-Desert",
    "dominant_province_description_Intermountain Semi-Desert and Desert",
    "dominant_province_description_Sierran Steppe-Mixed Forest-Coniferous Forest-Alpine Meadow",

    # Section
    "dominant_section_description_Central California Coast",
    "dominant_section_description_Central Valley Coast Ranges",
    "dominant_section_description_Colorado Desert",
    "dominant_section_description_Great Valley",
    "dominant_section_description_Klamath Mountains",
    "dominant_section_description_Modoc Plateau",
    "dominant_section_description_Mojave Desert",
    "dominant_section_description_Mono",
    "dominant_section_description_Northern California Coast",
    "dominant_section_description_Northern California Coast Ranges",
    "dominant_section_description_Northern California Interior Coast Ranges",
    "dominant_section_description_Northwestern Basin and Range",
    "dominant_section_description_Sierra Nevada",
    "dominant_section_description_Sierra Nevada Foothills",
    "dominant_section_description_Sonoran Desert",
    "dominant_section_description_Southeastern Great Basin",
    "dominant_section_description_Southern California Coast",
    "dominant_section_description_Southern California Mountains and Valleys",
    "dominant_section_description_Southern Cascades"
]

In [ ]:
coded_seasons =[
    'Season_Spring',
    'Season_Summer',
    'Season_Fall',
    'Season_Winter'
]

In [ ]:
# 3-day lagged features
lag_3_day = [
    "Actual Evapotranspiration 3 Day Sum",
    "Solar Radiation 3 Day Sum",
    "Daily Minimum Air Temperature 3 Day Mean",
    "Daily Maximum Air Temperature 3 Day Mean",
    "Vapor Pressure Deficit 3 Day Mean",
    "Wind Speed 3 Day Median",
    "log_Precipitation 3 Day Sum",
    "Maximum Relative Humidity 3 Day Mean",
    "Minimum Relative Humidity 3 Day Mean",
    "Specific Humidity 3 Day Mean",
    "100-hour Dead Fuel Moisture 3 Day Median",
    "1000-hour Dead Fuel Moisture 3 Day Median",
    "Burning Index 3 Day Mean",
    "Energy Release Component 3 Day Mean",
    "fire_count 3 Day Sum",
]

# 7-day lagged features
lag_7_day = [
    "Actual Evapotranspiration 7 Day Sum",
    "Solar Radiation 7 Day Sum",
    "Daily Minimum Air Temperature 7 Day Mean",
    "Daily Maximum Air Temperature 7 Day Mean",
    "Vapor Pressure Deficit 7 Day Mean",
    "Wind Speed 7 Day Median",
    "log_Precipitation 7 Day Sum",
    "Maximum Relative Humidity 7 Day Mean",
    "Minimum Relative Humidity 7 Day Mean",
    "Specific Humidity 7 Day Mean",
    "100-hour Dead Fuel Moisture 7 Day Median",
    "1000-hour Dead Fuel Moisture 7 Day Median",
    "Burning Index 7 Day Mean",
    "Energy Release Component 7 Day Mean",
    "fire_count 7 Day Sum",
]

# 30-day lagged features
lag_30_day = [
    "Actual Evapotranspiration 30 Day Sum",
    "Solar Radiation 30 Day Sum",
    "Daily Minimum Air Temperature 30 Day Mean",
    "Daily Maximum Air Temperature 30 Day Mean",
    "Vapor Pressure Deficit 30 Day Mean",
    "Wind Speed 30 Day Median",
    "log_Precipitation 30 Day Sum",
    "Maximum Relative Humidity 30 Day Mean",
    "Minimum Relative Humidity 30 Day Mean",
    "Specific Humidity 30 Day Mean",
    "100-hour Dead Fuel Moisture 30 Day Median",
    "1000-hour Dead Fuel Moisture 30 Day Median",
    "Burning Index 30 Day Mean",
    "Energy Release Component 30 Day Mean",
    "fire_count 30 Day Sum",
]


In [ ]:
feature_sets =  {
    "Water Demand": water_demand, 
    "Water Supply": water_supply, 
    "Water Supply Indexes": water_supply_indexes, 
    "Fire Danger": fire_danger,
    "Social": social, 
    "Infrastructure": infrastructure,
    "Elevation": elevation, 
    "WUI" : WUI, 
    "Ecoregion": ecoregion, 
    "Land Cover": land_cover, 
    "Interactions": interactions, 
    "Wind Slope": wind_slope,
    "Others": others,
    "Coded Ecoregions": coded_ecoregion,
    "Coded Seasons": coded_seasons,
    'Spatial': spatial,
    "Lag 3 Day Features": lag_3_day,
    "Lag 7 Day Features": lag_7_day,
    "Lag 30 Day Features": lag_30_day
}

In [ ]:
variable_selection_output = pd.concat([X, detail_data], axis=1)

## Export File

In [ ]:
with open('../data/processed/feature_sets.json', 'w') as f:
    json.dump(feature_sets, f, indent=4)


print("All datasets saved successfully to ../data/processed/")